In [3]:
import stanza

In [16]:
import os
import json
import time
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import stanza
from tqdm import tqdm

# === PATHS ===

BASE_DIR = "../JCDL_Code_2022_2025/Scientific_Novelty_Detection_2022_2025"

CACHE_DIR = os.path.join(BASE_DIR, "cache")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
TRIPLET_DIR = os.path.join(BASE_DIR, "Triplets", "Novel_Papers")
TEMP_PDF_DIR = os.path.join(BASE_DIR, "temp_pdf")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(TRIPLET_DIR, exist_ok=True)
os.makedirs(TEMP_PDF_DIR, exist_ok=True)

# === GROBID API ===
GROBID_URL = "http://localhost:8070/api/processFulltextDocument"

# === STANZA PIPELINE ===
import torch

use_gpu = torch.cuda.is_available()

nlp = stanza.Pipeline(
    "en",
    processors="tokenize",
    use_gpu=use_gpu
)

print("Using GPU for Stanza:", use_gpu)

2026-02-24 08:44:56 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-02-24 08:44:57 INFO: Downloaded file to C:\Users\spars\stanza_resources\resources.json
2026-02-24 08:44:57 WARNING: Language en package default expects mwt, which has been added
2026-02-24 08:44:57 INFO: Loading these models for language: en (English):
| Processor | Package  |
------------------------
| tokenize  | combined |
| mwt       | combined |

2026-02-24 08:44:57 INFO: Using device: cuda
2026-02-24 08:44:57 INFO: Loading: tokenize
2026-02-24 08:44:57 INFO: Loading: mwt
2026-02-24 08:44:57 INFO: Done loading processors!


Using GPU for Stanza: True


In [17]:
def load_metadata(task):
    path = os.path.join(CACHE_DIR, f"{task}_metadata.json")
    if not os.path.exists(path):
        print(f"No metadata for {task}")
        return []
    with open(path, "r") as f:
        return json.load(f)

In [18]:
def load_checkpoint(task):
    path = os.path.join(CHECKPOINT_DIR, f"{task}_processing_checkpoint.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"processed": [], "failed": []}

def save_checkpoint(task, data):
    path = os.path.join(CHECKPOINT_DIR, f"{task}_processing_checkpoint.json")
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

In [19]:
def download_pdf(pdf_url, paper_id):
    if not pdf_url:
        return None

    temp_path = os.path.join(TEMP_PDF_DIR, f"{paper_id}.pdf")

    try:
        r = requests.get(pdf_url, timeout=30)
        if r.status_code == 200:
            with open(temp_path, "wb") as f:
                f.write(r.content)
            return temp_path
    except Exception as e:
        print(f"Download failed: {e}")

    return None

In [20]:
def process_with_grobid(pdf_path):
    with open(pdf_path, "rb") as f:
        files = {"input": f}
        response = requests.post(GROBID_URL, files=files)

    if response.status_code == 200:
        return response.text
    return None

In [21]:
def extract_body_text(tei_xml):
    try:
        root = ET.fromstring(tei_xml)
    except:
        return ""

    texts = []

    for elem in root.iter():
        if elem.text:
            texts.append(elem.text.strip())

    return " ".join(texts)

In [22]:
def get_sentences(text):
    doc = nlp(text)
    sentences = []
    for sent in doc.sentences:
        s = " ".join([token.text for token in sent.tokens])
        sentences.append(s)
    return sentences

In [23]:
def extract_triplets_from_sentences(sentences, paper_id, task):
    rows = []

    for idx, sentence in enumerate(sentences):
        words = sentence.split()

        if len(words) > 3:
            sub = words[0]
            pred = words[1]
            obj = " ".join(words[2:5])

            rows.append({
                "topic": task,
                "paper_ID": paper_id,
                "sentence_ID": idx,
                "info-unit": "auto",
                "sub": sub,
                "pred": pred,
                "obj": obj,
                "triplets": f"{sub} {pred} {obj}",
                "pred_weights": None
            })

    return pd.DataFrame(rows)

In [24]:
def append_triplets(df, task):
    path = os.path.join(TRIPLET_DIR, f"{task}_triplets_results.csv")
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)

In [26]:
TASKS = ["Dia2022_2025", "MT2022_2025",
         "QA2022_2025", "SA2022_2025",
         "Sum2022_2025"]

for task in TASKS:

    print(f"\nProcessing Task: {task}")

    metadata = load_metadata(task)
    checkpoint = load_checkpoint(task)

    for paper in tqdm(metadata):

        paper_id = paper["id"].split("/")[-1]

        if paper_id in checkpoint["processed"]:
            continue

        try:
            pdf_path = download_pdf(paper.get("pdf_url"), paper_id)

            if not pdf_path:
                checkpoint["failed"].append(paper_id)
                save_checkpoint(task, checkpoint)
                continue

            tei_xml = process_with_grobid(pdf_path)

            os.remove(pdf_path)  # delete immediately

            if not tei_xml:
                checkpoint["failed"].append(paper_id)
                save_checkpoint(task, checkpoint)
                continue

            body_text = extract_body_text(tei_xml)
            sentences = get_sentences(body_text)

            df_triplets = extract_triplets_from_sentences(
                sentences, paper_id, task
            )

            if not df_triplets.empty:
                append_triplets(df_triplets, task)

            checkpoint["processed"].append(paper_id)
            save_checkpoint(task, checkpoint)

        except Exception as e:
            print(f"Error on {paper_id}: {e}")
            checkpoint["failed"].append(paper_id)
            save_checkpoint(task, checkpoint)
            continue


Processing Task: Dia2022_2025


  6%|▌         | 14/240 [00:24<04:22,  1.16s/it]

Download failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


 21%|██▏       | 51/240 [00:38<01:25,  2.20it/s]

Download failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


...

100%|██████████| 22/22 [00:15<00:00,  1.42it/s]



Processing Task: SA2022_2025


100%|██████████| 56/56 [00:44<00:00,  1.25it/s]



Processing Task: Sum2022_2025


100%|██████████| 12/12 [00:08<00:00,  1.34it/s]
